# Denoisify - Spectral Subtraction

# Example of Use

This Jupyter notebook provides an example of how to use the **Denoisify - Spectral Subtraction** function, which implements an iterative variant of the **Spectral Subtraction** technique for noise reduction in audio signals. This approach integrates **Spectral Modeling** based on **Sinusoidal Modeling** and **Transient Detection**, enhancing the accuracy of noise removal while preserving the essential characteristics of the original signal. Additionally, an algorithm for **Musical Noise** reduction has been incorporated to mitigate artifacts commonly introduced by spectral subtraction methods. In this notebook, the steps to apply this technique to a noisy audio signal are outlined, demonstrating its effectiveness in improving audio quality and reducing noise while maintaining the integrity of the processed signal.

## Imports

In [ ]:
from scripts.core import denoisify_ss
from scripts.utils import print_init

## Audio and Noise Signals  

The audio signal selected for this test is *Mother* by Pink Floyd, with a sampling rate of 44.1 kHz. The noise signal, also sampled at 44.1 kHz, was obtained from a **Revox A77** tape recorder and was sourced from the work of Irigaray et al., presented at the AES International Conference on Audio Archiving, Preservation & Restoration, Culpeper, VA, USA, June 2023:

I. Irigaray, M. Rocamora, and L. W. P. Biscainho, *Noise reduction in analog tape audio recordings with deep learning models*, AES International Conference on Audio Archiving, Preservation & Restoration, 2023.


In [ ]:
song_dir = 'Mother-Pink_Floyd.wav'

noise_dir = 'Noise.wav' 

duration = 30 

snr_db = 10

nfft = 2048

x, fs = print_init(song_dir, noise_dir, duration, snr_db, nfft)

## Parameters to Modify in Denoisify - Spectral Subtraction

### **Denoising / Spectral Subtraction**
- `--nfft NFFT`  
  FFT size for STFT analysis and synthesis. (default: 2048)

- `--n_iter N_ITER`  
  Number of iterations of spectral subtraction. (default: 28)

- `--alpha ALPHA`  
  Over-subtraction factor. (default: 0.65)

- `--beta BETA`  
  Spectral floor factor. (default: 0.01)

- `--rho RHO`  
  Low pass filter smoothing factor. (default: 0.01)

---

### **Noise Profile Detection**
- `--th_energy TH_ENERGY`  
  Scaling factor for energy-based silence threshold. (default: 0.75)

- `--th_zcr TH_ZCR`  
  Scaling factor for ZCR-based silence threshold. (default: 0.35)

- `--th_he TH_HE`  
  Scaling factor for high-frequency content threshold. (default: 0.05)

- `--zcr_hf_pct_cut ZCR_HF_PCT_CUT`  
  Fraction (0–1) of the ZCR used to define the high-frequency cut-off. (default: 0.9)

- `--min_silence_len MIN_SILENCE_LEN`  
  Minimum number of frames to consider a segment as silence. (default: 10)

- `--min_sound_len MIN_SOUND_LEN`  
  Minimum number of frames to separate two silence segments. (default: 25)

- `--start_silence START_SILENCE`  
  Number of frames removed from the start of each detected silence to avoid transients. (default: 8)

- `--end_silence END_SILENCE`  
  Number of frames removed from the end of each detected silence to avoid transients. (default: 1)

- `--num_init_frames NUM_INIT_FRAMES`  
  Number of ending frames assumed to be pure noise. (default: 5)

---

### **Spectral Modeling**
- `--sm_mode SM_MODE`  
  Spectral modeling mode (0: No modeling, 1: Transients + Sinusoids, 2: Transients only, 3: Sinusoids only). (default: 1)

- `--sm_keep_pct SM_KEEP_PCT`  
  Percentage of iterations to retain the spectral model (0 to 1). (default: 0.5)

- `--sm_nfft SM_NFFT`  
  FFT size for sinusoidal modeling. (default: 2048)

- `--peak_thresh PEAK_THRESH`  
  Threshold for peak detection in sinusoidal modeling. (default: -60)

- `--min_sine_dur MIN_SINE_DUR`  
  Minimum duration of a sinusoid. (default: 0.01)

- `--max_sines MAX_SINES`  
  Maximum number of sinusoids. (default: 100)

- `--fdev_offset FDEV_OFFSET`  
  Offset threshold for frequency deviation in peak continuation. (default: 20)

- `--fdev_slope FDEV_SLOPE`  
  Slope threshold for frequency deviation in peak continuation. (default: 0.01)

- `--td_nfft TD_NFFT`  
  FFT size for transient detection. (default: 2048)

- `--td_Lh TD_LH`  
  Horizontal median filter length given in seconds or frames. (default: 0.9)

- `--td_Lp TD_LP`  
  Percussive median filter length given in Hertz or bins. (default: 1000.0)

---

### **Musical Noise**
- `--remove_mn REMOVE_MN`  
  If True, apply musical noise reduction after spectral subtraction. (default: False)

- `--mn_nfft MN_NFFT`  
  FFT size for musical noise reduction. (default: 256)

- `--mn_thresh_db MN_THRESH_DB`  
  Threshold in dB for spectral floor in musical noise reduction. (default: -25)

- `--mn_win_len MN_WIN_LEN`  
  Window length for musical noise reduction. (default: 44)

---

In [ ]:
# Denoising / Spectral Subtraction #
nfft = 2048
n_iter = 30
alpha = 0.10  # alpha = 0.86, 0.99, 0.10
beta = 0.93   # beta  = 0.01, 0.93
rho = 0.01    # rho   = 0.01, 0.15 

# Noise Profile Detection #
th_energy = 0.75
th_zcr = 0.35
th_he = 0.05    
zcr_hf_pct_cut = 0.9
min_silence_len = 10
min_sound_len = 25
start_silence = 8 
end_silence = 1
num_init_frames = 5

# Spectral Modeling #
sm_mode = 1
sm_keep_pct = 0.50
sm_nfft = 2048
peak_thresh = -60
min_sine_dur = 0.01 
max_sines = 100                                                    
fdev_offset = 20 
fdev_slope = 0.01
td_nfft = 2048 
td_Lh = 0.9
td_Lp = 1000

# Musical Noise #
remove_mn = False  
mn_nfft = 256
mn_thresh_db = -25
mn_win_len = 44 

# Debugging #
debug = True

In [ ]:
# SIGNAL DENOISING #

y = denoisify_ss(
        x, fs,        

        nfft,  
        n_iter,
        alpha, beta, rho, 

        th_energy, th_zcr, th_he,        
        zcr_hf_pct_cut,
        min_silence_len, min_sound_len, 
        start_silence, end_silence, 
        num_init_frames,

        sm_mode, 
        sm_keep_pct,  
        sm_nfft, peak_thresh, min_sine_dur, max_sines, fdev_offset, fdev_slope,
        td_nfft, td_Lh, td_Lp, 

        remove_mn, mn_nfft, mn_thresh_db, mn_win_len,
        
        debug                              
)